In [30]:
# reading the data as a list of lines 

with open('dictionary.txt', 'r') as f:
    file = f.readlines()
    data = [line.rstrip() for line in file]

len(data)

491775

In [63]:
#some data cleaning

def is_amharic_word(word):
    
    for char in word:
        if not (ord('\u1200') <= ord(char) <= ord('\u137F')):
            return False
    
    return True

In [9]:
# the first 10 words sorted by frequency  

data[:10]

['ነው 260338',
 'ላይ 174891',
 'ውስጥ 80072',
 'እና 77266',
 'ወደ 75254',
 'ጋር 70525',
 'ነበር 69074',
 'ጊዜ 64423',
 'ግን 62811',
 'ነገር 57927']

In [31]:
# creating a dictionary of counts 

word_frequency = dict([(word.split()[0] , int(word.split()[-1])) for word in data])


word_frequency = dict([(word, cnt) for word, cnt in word_frequency.items() if is_amharic_word(word)])




In [33]:
# probability of each word --> frequentiest approach 

total = float(sum(word_frequency.values()))
probs = {word:(cnt/ total) for word, cnt in word_frequency.items()}


In [34]:
# top 10 most probable words

sorted(probs.items() , key = lambda x:-x[1])[:10]


[('ነው', 0.01556774136373601),
 ('ላይ', 0.010458165365198913),
 ('ውስጥ', 0.004788160723663352),
 ('እና', 0.004620367000631589),
 ('ወደ', 0.004500053040995128),
 ('ጋር', 0.0042172673972969065),
 ('ነበር', 0.004130500222628664),
 ('ጊዜ', 0.003852378837803029),
 ('ግን', 0.0037559841544362424),
 ('ነገር', 0.003463929791183522)]

In [57]:
# class for spell checking and correction system 

class SpellCheck:
    
    def __init__(self, dictionary, character_set, probs):
        
        self.dictionary = dictionary 
        self.all_letters = character_set
        self.probs = probs
        
    def split(self, word):
        return [(word[:i], word[i:]) for i in range(len(word) + 1)]
    
    def delete(self, word):
        return [l + r[1:] for l,r in self.split(word) if r]
    
    def insert(self, word):
        return [l + c + r for l, r in self.split(word) for c in self.all_letters]
    
    def swap(self, word):
        return [l + r[1] + r[0] + r[2:] for l, r in self.split(word) if len(r)>1]
    
    def replace(self, word):
        return [l + c + r[1:] for l, r in self.split(word) if r for c in self.all_letters]

    def get_all_edits(self, word):
        
        return self.delete(word) + self.insert(word) + self.replace(word) + self.swap(word)
    
    def edits(self, word, distance = 2):
        
        res = []
        all_edits = [word]
        
        for _ in range(distance):
            
            next_edit = []
            
            for w in all_edits:
                next_edit.extend(self.get_all_edits(w))
            all_edits = next_edit
            res.extend(all_edits)
        
        return set(res)
    
    def suggestion(self, word):
        
        if word in self.dictionary:
            
            print(f'{word} is correctly spelled')
            
            return 
        
        word_edits = self.edits(word)
        
        suggestions = [(w, self.probs[w]) for w in word_edits if w in self.probs]
        
        return sorted(suggestions , key = lambda x:-x[1])[:10]
    
    
    
        
    
    

In [58]:
dictionary = set(word_frequency.keys())
character_set = set("".join(word_frequency.keys()).replace(" ", ""))


spl = SpellCheck(dictionary, character_set, probs)

In [62]:
spl.suggestion('አናቆት')

[('አባላት', 0.0005986396868392674),
 ('አለበት', 0.00044992158663256896),
 ('አይነት', 0.0004420880236542507),
 ('አመት', 0.0003211760821110483),
 ('አምስት', 0.0003128641336073366),
 ('አራት', 0.00029456588726103596),
 ('አካላት', 0.000280692707024625),
 ('ናት', 0.0002711847947074296),
 ('አገራት', 0.000271005400135407),
 ('ጥናት', 0.00026359042449180804)]